# Preparación de datos — PlantVillage (sesión 4, bloque 2)

**Candidato, nada congelado.** Sin modelos ni métricas. No se escribe en `data/splits/`: el candidato vive en `DATA_ROOT/interim/` (no versionado) y el congelamiento lo hace el equipo después.
Config: `configs/01_preparacion_datos.yaml`. Rutas: `configs/paths.yaml` (`DATA_ROOT`, sobreescribible por variable de entorno).

In [1]:
import subprocess, sys, time
import numpy as np, pandas as pd, yaml
from foliares.utils.paths import cargar_rutas
from foliares.utils.seeds import fijar_semillas
from foliares.data.manifiesto_pv import construir_manifiesto
from foliares.data import particion_pv as pp
try:
    import psutil; _proc = psutil.Process()
except ImportError:
    _proc = None
def mem_mb(): return round(_proc.memory_info().rss / 2**20) if _proc else None

pd.set_option("display.width", 220, "display.max_rows", 200, "display.max_columns", 40)
R = cargar_rutas()
CFG = yaml.safe_load((R.repo / "configs" / "01_preparacion_datos.yaml").read_text(encoding="utf-8"))
SEMILLA = CFG["semilla"]; print("semillas:", fijar_semillas(SEMILLA))
h = subprocess.check_output(["git", "-C", str(R.plantvillage), "log", "-1", "--format=%H"], text=True).strip()
print("PlantVillage", h, "OK" if h == CFG["commit_plantvillage"] else "NO COINCIDE")
INTERIM = R.interim; INTERIM.mkdir(parents=True, exist_ok=True)
T0 = time.time()

semillas: {'semilla': 42, 'python': True, 'numpy': True, 'torch': False, 'cuda': False}
PlantVillage 7f7ecc7e1eaca78107e3affe7cb5abd9427e139a OK


## 2a. Manifiesto (15 clases candidatas)

`clase_comun` = tiene par en PlantDoc según `docs/mapeo_clases.md` (borrador); **no implica decisión**. La hoja es `leaf_id = clase:::hoja`; vacío si no hay grupo. Rutas relativas a `DATA_ROOT`.

In [2]:
t = time.time()
man, ctrl = construir_manifiesto(R)
print(f"{len(man)} imágenes, {man.clase.nunique()} clases — {time.time()-t:.1f} s, RSS {mem_mb()} MB")
print("\nPar color/segmented (convención: <stem de color>_final_masked.jpg):")
display(ctrl["par_color_segmented"])
tot = ctrl["par_color_segmented"].sum(numeric_only=True)
print(f"Color sin par segmented: {tot.color_sin_par} | segmented sin color: {tot.segmented_sin_color} | pares con convención irregular (segmented sin uuid, enlazados a mano): {tot.par_irregular}")
display(man[man.segmented_irregular][["ruta_color", "ruta_segmented"]].T)
man.to_csv(INTERIM / CFG["salida"]["manifiesto"], index=False)
print("Guardado:", R.relativa(INTERIM / CFG["salida"]["manifiesto"]))

22787 imágenes, 15 clases — 1.4 s, RSS 156 MB

Par color/segmented (convención: <stem de color>_final_masked.jpg):


,clase,color,segmented_en_disco,par_irregular,color_sin_par,segmented_sin_color
0,Tomato___Bacterial_spot,2127,2127,0,0,0
1,Tomato___Early_blight,1000,1000,0,0,0
2,Tomato___Late_blight,1909,1909,0,0,0
3,Tomato___Leaf_Mold,952,952,0,0,0
4,Tomato___Septoria_leaf_spot,1771,1771,0,0,0
5,Tomato___Spider_mites Two-spotted_spider_mite,1676,1676,1,0,0
6,Tomato___Target_Spot,1404,1404,0,0,0
7,Tomato___Tomato_Yellow_Leaf_Curl_Virus,5357,5357,0,0,0
8,Tomato___Tomato_mosaic_virus,373,373,0,0,0
9,Tomato___healthy,1591,1591,0,0,0


Color sin par segmented: 0 | segmented sin color: 0 | pares con convención irregular (segmented sin uuid, enlazados a mano): 1


,7844
ruta_color,raw/plantvillage/raw/color/Tomato___Spider_mit...
ruta_segmented,raw/plantvillage/raw/segmented/Tomato___Spider...


Guardado: interim/manifiesto_plantvillage.csv


In [3]:
res = man.groupby("clase").agg(imagenes=("archivo" if "archivo" in man else "ruta_color", "size"), con_grupo=("tiene_grupo", "sum"),
                              sesiones=("sesion", "nunique"), clase_comun=("clase_comun", "first"),
                              par_plantdoc=("clase_plantdoc_candidata", "first"))
res["sin_grupo"] = res.imagenes - res.con_grupo
res["hojas"] = man[man.tiene_grupo].groupby("clase").leaf_id.nunique()
display(res.fillna({"hojas": 0}).astype({"hojas": int}))
print("Total:", len(man), "| con grupo:", int(man.tiene_grupo.sum()), "| sin grupo:", int((~man.tiene_grupo).sum()), "| clase_comun=True:", man[man.clase_comun].clase.nunique(), "clases")

,imagenes,con_grupo,sesiones,clase_comun,par_plantdoc,sin_grupo,hojas
clase,,,,,,,
"Pepper,_bell___Bacterial_spot",997,997,2,True,Bell_pepper leaf spot,0,175
"Pepper,_bell___healthy",1478,1476,3,True,Bell_pepper leaf,2,165
Potato___Early_blight,1000,1000,1,True,Potato leaf early blight,0,250
Potato___Late_blight,1000,1000,1,True,Potato leaf late blight,0,250
Potato___healthy,152,152,1,False,,0,38
Tomato___Bacterial_spot,2127,2127,2,True,Tomato leaf bacterial spot,0,532
Tomato___Early_blight,1000,999,1,True,Tomato Early blight leaf,1,250
Tomato___Late_blight,1909,920,6,True,Tomato leaf late blight,989,230
Tomato___Leaf_Mold,952,952,1,True,Tomato mold leaf,0,238


Total: 22787 | con grupo: 16036 | sin grupo: 6751 | clase_comun=True: 13 clases


## 2b. Mosaic virus: zona de descarte, validada antes de aplicarla

Mosaic no tiene ningún grupo y es una sola tanda numerada (`PSU_CG`, 2047–2420). Propuesta: cortes por rangos contiguos con una zona de descarte de N números entre train, val y test.

**Validación** sobre las tandas (clase, sesión) que sí tienen grupo y numeración: se sortean `n_cortes_por_unidad` pares de cortes al azar (cada tramo ≥ 5 % de la tanda), se aplica la zona y se cuenta en qué proporción de cortes alguna hoja real queda con imágenes en ≥ 2 tramos. **Criterio fijado de antemano:** proporción agregada ≤ 1 % → se aplica; si no, se prueban 15 y 20; si ninguna, mosaic va solo a train.

In [4]:
t = time.time(); V = CFG["validacion_zona"]
por_unidad, resumen, zona = pp.validar_zonas(man, tuple(V["zonas"]), V["n_cortes_por_unidad"], SEMILLA, V["criterio_max_prop_con_fuga"])
print(f"{por_unidad.unidad.nunique()} tandas con grupo; {time.time()-t:.1f} s, RSS {mem_mb()} MB")
r = resumen.copy()
for c in ("prop_con_fuga", "wilson95_bajo", "wilson95_alto", "peor_unidad_prop"): r[c] = (100 * r[c]).round(2)
display(r.rename(columns={"prop_con_fuga": "% cortes con fuga", "wilson95_bajo": "IC95 bajo %", "wilson95_alto": "IC95 alto %", "peor_unidad_prop": "peor tanda %"}))
print("Zona elegida por la regla fijada de antemano:", zona)

16 tandas con grupo; 22.7 s, RSS 166 MB


,zona,unidades,cortes,cortes_con_fuga,% cortes con fuga,IC95 bajo %,IC95 alto %,peor_unidad,peor tanda %,cumple
0,10,16,160000,7026,4.39,4.29,4.49,Tomato___Bacterial_spot|UF.GRC_BS_Lab Leaf,42.39,False
1,15,16,160000,2834,1.77,1.71,1.84,Tomato___Bacterial_spot|UF.GRC_BS_Lab Leaf,23.48,False
2,20,16,160000,1342,0.84,0.80,0.88,Tomato___Bacterial_spot|UF.GRC_BS_Lab Leaf,11.25,True


Zona elegida por la regla fijada de antemano: 20


In [5]:
piv = por_unidad.pivot_table(index=["clase", "sesion"], columns="zona", values="prop_con_fuga").mul(100).round(2)
piv["imagenes"] = por_unidad.drop_duplicates("unidad").set_index(["clase", "sesion"]).imagenes
piv["hojas"] = por_unidad.drop_duplicates("unidad").set_index(["clase", "sesion"]).hojas
display(piv.rename(columns={10: "% fuga z=10", 15: "% fuga z=15", 20: "% fuga z=20"}))
por_unidad.to_csv(INTERIM / CFG["salida"]["validacion_por_unidad"], index=False)
mos = man[man.clase.isin(CFG["clases_por_rango"])]
print("Mosaic con cada zona (el candidato usa la elegida):")
filas = []
for z in V["zonas"]:
    lab = pd.Series(pp.cortar_por_rangos(mos.num.to_numpy(), z)).map({0: "train", 1: "val", 2: "test", -1: "descartadas"})
    filas.append({"zona": z, **lab.value_counts().to_dict()})
display(pd.DataFrame(filas).set_index("zona")[["train", "val", "test", "descartadas"]])

zona                                                              % fuga z=10  % fuga z=15  % fuga z=20  imagenes  hojas
clase                                         sesion                                                                    
Pepper,_bell___Bacterial_spot                 JR_B.Spot                  7.21         0.68         0.00       589     73
                                              NREC_B.Spot                0.00         0.00         0.00       408    102
Pepper,_bell___healthy                        JR_HL                     10.95         1.02         0.00      1476    165
Potato___Early_blight                         RS_Early.B                 0.21         0.00         0.00      1000    250
Potato___Late_blight                          RS_LB                      0.21         0.00         0.00      1000    250
Potato___healthy                              RS_HL                      0.76         0.00         0.00       152     38
Tomato___Bacterial_spot                       GCREC_Bact.Sp              0.00         0.00         0.00      1756    439
                                              UF.GRC_BS_Lab Leaf        42.39        23.48        11.25       371     93
Tomato___Early_blight                         RS_Erly.B                  0.92         0.79         0.94       999    250
Tomato___Late_blight                          RS_Late.B                  0.00         0.00         0.00       920    230
Tomato___Leaf_Mold                            Crnl_L.Mold                0.00         0.00         0.00       952    238
Tomato___Septoria_leaf_spot                   Matt.S_CG                  7.61         2.37         1.23      1000    250
Tomato___Spider_mites Two-spotted_spider_mite Com.G_SpM_FL               0.00         0.00         0.00      1676    419
Tomato___Tomato_Yellow_Leaf_Curl_Virus        YLCV_GCREC                 0.00         0.00         0.00      1470    368
                                              YLCV_NREC                  0.00         0.00         0.00      1268    317
Tomato___healthy                              RS_HL                      0.00         0.00         0.00       999    250

Mosaic con cada zona (el candidato usa la elegida):


,train,val,test,descartadas
zona,,,,
10,262,46,45,20
15,262,41,40,30
20,262,36,35,40


## 2c. Candidato de particiones (por hoja, por clase, 70/15/15)

Sin grupo → solo train (salvo mosaic: rangos). Tope de train por clase con muestreo estratificado por sesión. Guardado en `DATA_ROOT/interim/` como **CANDIDATO**.

In [6]:
cand = pp.construir_candidato(man, SEMILLA, tuple(CFG["fracciones"]), CFG["tope_train_por_clase"], tuple(CFG["clases_por_rango"]), zona)
cand.to_csv(INTERIM / CFG["salida"]["candidato"], index=False)
print("Guardado:", R.relativa(INTERIM / CFG["salida"]["candidato"]), "| zona usada para mosaic:", zona)
print(cand.groupby(["particion", "motivo_exclusion"]).size())
print("Contenido de SPLITS_DIR:", sorted(p.name for p in R.splits.iterdir()))

Guardado: interim/particion_candidato_plantvillage.csv | zona usada para mosaic: 20
particion  motivo_exclusion
excluida   tope_train           3170
           zona_descarte          40
test                            2446
train                          14648
val                             2483
dtype: int64
Contenido de SPLITS_DIR: ['.gitkeep']


In [7]:
tab = pp.tabla_resumen(cand)
tot = tab.drop(columns=["clase", "clase_comun"]).sum(); tot["clase"] = "TOTAL"
display(pd.concat([tab, tot.to_frame().T], ignore_index=True))

,clase,total,clase_comun,img_train,hojas_train,img_val,hojas_val,img_test,hojas_test,excl_tope,excl_zona
0,"Pepper,_bell___Bacterial_spot",997,True,695,123,171,26,131,26,0,0
1,"Pepper,_bell___healthy",1478,True,1017,115,228,25,233,25,0,0
2,Potato___Early_blight,1000,True,696,174,152,38,152,38,0,0
3,Potato___Late_blight,1000,True,696,174,152,38,152,38,0,0
4,Potato___healthy,152,False,104,26,24,6,24,6,0,0
5,Tomato___Bacterial_spot,2127,True,1488,372,320,80,319,80,0,0
6,Tomato___Early_blight,1000,True,696,174,152,38,152,38,0,0
7,Tomato___Late_blight,1909,True,1500,162,136,34,136,34,137,0
8,Tomato___Leaf_Mold,952,True,664,166,144,36,144,36,0,0
9,Tomato___Septoria_leaf_spot,1771,True,1467,174,152,38,152,38,0,0


In [8]:
# Composición por sesión: ¿val/test ven las mismas sesiones que train? (por clase con >1 sesión)
sub = cand[cand.particion.isin(["train", "val", "test"])]
ct = sub.groupby(["clase", "sesion", "particion"]).size().unstack("particion", fill_value=0)
multi = ct.groupby("clase").filter(lambda g: len(g) > 1)
display(multi[["train", "val", "test"]])
print("Sesiones presentes en train y ausentes de val/test:")
aus = ct[(ct.train > 0) & (ct.val == 0) & (ct.test == 0)]
display(aus[["train"]].rename(columns={"train": "imgs_train"}))

particion                                                  train  val  test
clase                                  sesion                              
Pepper,_bell___Bacterial_spot          JR_B.Spot             415  119    55
                                       NREC_B.Spot           280   52    76
Pepper,_bell___healthy                 JR_HL                1015  228   233
                                       Screen Shot             1    0     0
                                       bell-pepper-plant-      1    0     0
Tomato___Bacterial_spot                GCREC_Bact.Sp        1248  252   256
                                       UF.GRC_BS_Lab Leaf    240   68    63
Tomato___Late_blight                   GHLB                  706    0     0
                                       GHLB Leaf              60    0     0
                                       GHLB_PS Leaf          136    0     0
                                       GHLB_PS leaf            2    0     0
                                       GH_HL Leaf              2    0     0
                                       RS_Late.B             594  136   136
Tomato___Septoria_leaf_spot            JR_Sept.L.S           497    0     0
                                       Keller.St_CG          274    0     0
                                       Matt.S_CG             696  152   152
Tomato___Tomato_Yellow_Leaf_Curl_Virus UF.GRC_YLCV_Lab       867    0     0
                                       YLCV_GCREC            338  228   220
                                       YLCV_NREC             295  184   192
Tomato___healthy                                               1    0     0
                                       CG                      1    0     0
                                       GH_HL Leaf            590    0     0
                                       RS_HL                 695  152   152

Sesiones presentes en train y ausentes de val/test:


particion                                                  imgs_train
clase                                  sesion                        
Pepper,_bell___healthy                 Screen Shot                  1
                                       bell-pepper-plant-           1
Tomato___Late_blight                   GHLB                       706
                                       GHLB Leaf                   60
                                       GHLB_PS Leaf               136
                                       GHLB_PS leaf                 2
                                       GH_HL Leaf                   2
Tomato___Septoria_leaf_spot            JR_Sept.L.S                497
                                       Keller.St_CG               274
Tomato___Target_Spot                   Com.G_TgS_FL              1404
Tomato___Tomato_Yellow_Leaf_Curl_Virus UF.GRC_YLCV_Lab            867
Tomato___healthy                                                    1
                                       CG                           1
                                       GH_HL Leaf                 590

## 2d. Tests de no-fuga (pytest)

`tests/test_particion_pv.py`: ninguna hoja en más de una partición; val y test solo con grupo (más mosaic); sin imágenes de PlantDoc en el manifiesto (ruta y bytes); reproducibilidad con la semilla; zona de descarte respetada; tope.

In [9]:
out = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_particion_pv.py", "-p", "no:cacheprovider", "--color=no"], cwd=R.repo, capture_output=True, text=True)
print(out.stdout[-1500:], out.stderr[-500:])
assert out.returncode == 0

................                                                         [100%]
16 passed in 6.72s
 


In [10]:
print(f"Tiempo total del notebook: {time.time()-T0:.0f} s | RSS: {mem_mb()} MB")

Tiempo total del notebook: 32 s | RSS: 175 MB
